# CS781 Phase 1 - one model on BOTH T4s (DDP)
Set `MODEL`, `MB`, `EXTRA` below, then **Save Version -> Save & Run All** (accelerator GPU T4 x2).
1. a 20-step DDP check (same flags). If it fails the notebook stops - the long run is NOT started.
2. the real run: trains up to `MAX_HOURS`, then writes `final.pt` + `DONE` and exits cleanly.
Outputs in `/kaggle/working/run_<model name>/` (latest.pt every 30 min, best.pt, final.pt, metrics.jsonl, train_log.txt).

In [1]:
import os, sys, json, re, glob, shutil, subprocess, time, threading
DATA = "/kaggle/input/datasets/mrinaldevnath/cs781-legal-data"
CODE_SRC = "/kaggle/input/datasets/mrinaldevnath/cs781-p1-train"
WORK = "/kaggle/working/cs781/train"

def find(root, name):
    hits = glob.glob(os.path.join(root, "**", name), recursive=True)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(sorted(hits, key=len)[0])
DATA = find(DATA, "manifest.json")
CODE_SRC = find(CODE_SRC, "train.py")
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(CODE_SRC, WORK)
os.chdir(WORK)
print("data :", DATA, os.listdir(DATA))
print("code :", WORK, sorted(os.listdir(WORK)))
for f in ["sampler.py", "train.py", "generate.py", "tests.py", "configs/e.json"]:
    assert os.path.exists(f), f"MISSING {f} - re-upload the latest train_code.zip as the dataset"
assert "test_sampler" in open("tests.py").read(), "tests.py is an OLD version - re-upload the latest code"
assert "smoke_steps" in open("train.py").read(), "train.py is an OLD version - re-upload the latest code"
os.makedirs("/kaggle/working/logs", exist_ok=True)

def run_parallel(jobs, timeout=None):
    """jobs: list of (tag, shell_cmd, gpu_index_or_None). Runs them at the same time (one per GPU), streams
    output prefixed with the tag, saves /kaggle/working/logs/<tag>.txt, returns {tag: (exit_code, text)}."""
    procs, outs, lock = {}, {}, threading.Lock()
    def reader(tag, p):
        buf = []
        for line in p.stdout:
            buf.append(line)
            with lock:
                print(f"[{tag}] {line}", end="", flush=True)
        outs[tag] = buf
    threads = []
    for tag, cmd, gpu in jobs:
        env = dict(os.environ)
        if gpu is not None:
            env["CUDA_VISIBLE_DEVICES"] = str(gpu)
        print(f"$ [{tag}] (GPU {gpu}) {cmd}", flush=True)
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env=env, start_new_session=True)
        procs[tag] = p
        t = threading.Thread(target=reader, args=(tag, p), daemon=True); t.start(); threads.append(t)
    timer = None
    if timeout:
        def kill():
            print("TIMEOUT - killing", flush=True)
            for p in procs.values():
                try: os.killpg(p.pid, 9)
                except Exception: pass
        timer = threading.Timer(timeout, kill); timer.start()
    for p in procs.values(): p.wait()
    for t in threads: t.join()
    if timer: timer.cancel()
    res = {}
    for tag, p in procs.items():
        txt = "".join(outs.get(tag, []))
        open(f"/kaggle/working/logs/{tag}.txt", "w").write(txt)
        res[tag] = (p.returncode, txt)
        print(f"[{tag}] exit {p.returncode}", flush=True)
    return res

data : /kaggle/input/datasets/mrinaldevnath/cs781-legal-data ['tokenizer', 'manifest.json', 'p1_loader.py', 'prep_p1.py', 'legal_bpe_v2.py', 'val', 'train', 'dev', 'dev_prompts.jsonl']
code : /kaggle/working/cs781/train ['README.md', 'attn_core.py', 'attn_cross.py', 'attn_gqa_local.py', 'attn_mha.py', 'attn_mla.py', 'configs', 'gated_deltanet.py', 'generate.py', 'kv_cache.py', 'layers.py', 'models.py', 'p1_loader.py', 'sampler.py', 'tests.py', 'train.py']


In [2]:
MODEL = "c"                 # a, b, c, d, e
MB = 16                      # per-GPU micro-batch (a: 16 is fine; d: 8 max)
EXTRA = "--compile"         # "--compile" for a,b,c,e (2.4x faster on b). For d use it only if the speed test said so, else ""
MAX_HOURS = 11.0            # wall-clock limit of the training script (Kaggle session limit is 12 h)
RESUME = None               # e.g. "/kaggle/input/<old-output>/run_b_gpt_mha/latest.pt" to continue after a crash

cfg = json.load(open(f"configs/{MODEL}.json")); NAME = cfg["name"]
TORCHRUN = "torchrun --standalone --nproc_per_node=2"
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB



## 1. DDP check (20 steps, real data, with the same flags)

In [3]:
# r = run_parallel([("ddp_check", f"{TORCHRUN} train.py --config configs/{MODEL}.json --data_root {DATA} --smoke --smoke_steps 20 --micro_batch {MB} {EXTRA}", None)], timeout=1500)
# rc, txt = r["ddp_check"]
# losses = [float(x) for x in re.findall(r"step \d+/\d+ loss ([\d.]+)", txt)]
# ok = rc == 0 and "EVAL step" in txt and "DONE" in txt and len(losses) >= 2 and losses[-1] < losses[0]
# print("check:", "OK" if ok else "FAILED", losses, re.findall(r"([\d,]+) tok/s", txt)[-1:])
# shutil.rmtree(f"/kaggle/working/run_{NAME}_smoke", ignore_errors=True)
# assert ok, "DDP check failed - training NOT started. see logs/ddp_check.txt"

## 2. Real run

In [4]:
cmd = (f"{TORCHRUN} train.py --config configs/{MODEL}.json --data_root {DATA} --out /kaggle/working/run_{NAME} "
       f"--micro_batch {MB} --max_hours {MAX_HOURS} {EXTRA}" + (f" --resume {RESUME}" if RESUME else ""))
res = run_parallel([(f"train_{MODEL}", cmd, None)])
rc, txt = res[f"train_{MODEL}"]
print("exit code", rc, [l for l in txt.splitlines() if "DONE" in l][-1:])

$ [train_c] (GPU None) torchrun --standalone --nproc_per_node=2 train.py --config configs/c.json --data_root /kaggle/input/datasets/mrinaldevnath/cs781-legal-data --out /kaggle/working/run_c_local_gqa_global_mha --micro_batch 16 --max_hours 11.0 --compile
[train_c] W1006 02:33:52.106000 49 torch/distributed/run.py:851] 
[train_c] W1006 02:33:52.106000 49 torch/distributed/run.py:851] *****************************************
[train_c] W1006 02:33:52.106000 49 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
[train_c] W1006 02:33:52.106000 49 torch/distributed/run.py:851] *****************************************
[train_c] [10-06 02:33:54] config configs/c.json  world=2  device=cuda:0  out=/kaggle/working/run_c_local_gqa_global_mha
[train_c] [10-06 02:33:56] model c_local_gqa_global_mha: 103.04M par

In [5]:
try:
    import pandas as pd, matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    df = pd.read_json(f"/kaggle/working/run_{NAME}/metrics.jsonl", lines=True)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    tr = df[df["loss"].notna()]; ax[0].plot(tr["step"], tr["loss"]); ax[0].set_title("train loss")
    ev = df[df["eval"].notna()]; ax[1].plot(ev["step"], [e["avg"] for e in ev["eval"]], marker="o"); ax[1].set_title("val loss (shared metric)")
    plt.savefig("/kaggle/working/curves.png", dpi=110)
except Exception as e:
    print("plot skipped:", e)